# Data Pipeline to Monitor Local Crime Trends
**PAI102 — Shreya Das**

Extract → Validate → Transform → Load (PostgreSQL) → Monitor, using the Cambridge Police Daily Log (Socrata API).

Run the cells top to bottom in Google Colab.

## 1. Install packages

In [ ]:
!pip install -q sodapy python-dotenv psycopg2-binary prefect

## 2. Set up PostgreSQL in Colab
Docker doesn't run in Colab, so we install Postgres directly (port 5432).

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y postgresql postgresql-contrib > /dev/null
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'password';"
!sudo -u postgres psql -c "CREATE DATABASE cpd_db;" || true

## 3. Configuration
Get a free Socrata app token at https://data.cambridgema.gov/profile/edit/developer_settings (optional, but it avoids throttling).

In [ ]:
%%writefile .env
SOCRATA_APP_TOKEN=
POSTGRES_HOST=localhost
POSTGRES_PORT=5432
POSTGRES_DB=cpd_db
POSTGRES_USER=postgres
POSTGRES_PASSWORD=password


Shared settings

In [ ]:
%%writefile config.py
"""Shared settings for the pipeline, read from a .env file / environment variables."""
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:  # python-dotenv not installed: fall back to plain environment variables
    pass

# --- Socrata (data source) ---
SOCRATA_DOMAIN = "data.cambridgema.gov"
DATASET_ID = "3gki-wyrb"  # Cambridge Police Daily Log
SOCRATA_APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")  # optional, avoids throttling

# --- PostgreSQL (destination) ---
# Defaults match the docker-compose setup (host port 5433 -> container 5432).
# In Google Colab (apt-installed Postgres) set POSTGRES_PORT=5432.
PG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5433")),
    "dbname": os.getenv("POSTGRES_DB", "cpd_db"),
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "password"),
}

TABLE = "cpd_incidents"
REPORT_DIR = os.getenv("REPORT_DIR", "reports")


def get_connection():
    import psycopg2
    return psycopg2.connect(**PG)


Prefect decorators (falls back to plain functions if Prefect is missing)

In [ ]:
%%writefile prefect_compat.py
"""Use Prefect's @task / @flow if Prefect is installed; otherwise fall back to no-op
decorators so every step can still be run and tested as plain Python."""
try:
    from prefect import task, flow  # noqa: F401
    HAS_PREFECT = True
except ImportError:
    HAS_PREFECT = False

    def _passthrough(*dargs, **dkwargs):
        if len(dargs) == 1 and callable(dargs[0]) and not dkwargs:
            return dargs[0]
        return lambda fn: fn

    task = flow = _passthrough


## 4. Extract

In [ ]:
%%writefile extract.py
"""E — Extract: download the Cambridge Police Daily Log via the Socrata Open Data API."""
import pandas as pd

from config import SOCRATA_DOMAIN, DATASET_ID, SOCRATA_APP_TOKEN
from prefect_compat import task


@task(retries=3, retry_delay_seconds=[10, 10, 10])  # retry the API request if it fails
def extract_data() -> pd.DataFrame:
    """Return every incident in the dataset as a pandas DataFrame."""
    from sodapy import Socrata

    if not SOCRATA_APP_TOKEN:
        print("Warning: SOCRATA_APP_TOKEN not set - requests may be throttled.")

    # timeout raised from the 10 s default; fetching everything can take longer
    client = Socrata(SOCRATA_DOMAIN, SOCRATA_APP_TOKEN, timeout=30)
    try:
        results = client.get_all(DATASET_ID)  # paginates over all records
        df = pd.DataFrame.from_records(results)
    finally:
        client.close()

    print(f"Extracted {len(df):,} rows")
    return df


## 5. Validate

In [ ]:
%%writefile validate.py
"""V — Validate: sanity-check the raw data before transforming it.

Fail loudly (raise) when the data can't be trusted at all:
  * schema differs from what we expect
  * a date_time is not ISO 8601
  * a required column (date_time, id, type, location) has missing values
Fail softly for non-numeric IDs: they become NaN and are dropped in transform.
"""
from datetime import datetime

import pandas as pd

from prefect_compat import task

SCHEMA_COLS = ["date_time", "id", "type", "subtype", "location", "last_updated", "description"]
REQUIRED_COLS = ["date_time", "id", "type", "location"]


def check_valid_schema(df: pd.DataFrame) -> None:
    missing = set(SCHEMA_COLS) - set(df.columns)
    extra = set(df.columns) - set(SCHEMA_COLS)
    if missing:
        raise ValueError(f"Schema mismatch - missing columns: {sorted(missing)}")
    if extra:
        # New columns from the API shouldn't crash the pipeline; ignore them.
        print(f"Note: ignoring unexpected columns {sorted(extra)}")


def check_numeric_id(df: pd.DataFrame) -> pd.DataFrame:
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    bad = int(df["id"].isna().sum())
    if bad:
        print(f"Note: {bad} rows have non-numeric IDs and will be removed in transform")
    return df


def verify_datetime(df: pd.DataFrame) -> None:
    bad = []
    for value in df["date_time"].dropna().astype(str):
        try:
            datetime.fromisoformat(value)
        except ValueError:
            bad.append(value)
    if bad:
        raise ValueError(f"{len(bad)} date_time values are not ISO 8601, e.g. {bad[:3]}")


def check_missing_values(df: pd.DataFrame) -> None:
    for col in REQUIRED_COLS:
        n = int(df[col].isna().sum())
        if n:
            raise ValueError(f"{n} missing values in required column '{col}'")


@task
def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        raise ValueError("No data was extracted.")
    check_valid_schema(df)
    df = df[SCHEMA_COLS].copy()
    # Check for genuinely missing values BEFORE coercing IDs: coercion turns
    # non-numeric IDs into NaN on purpose (dropped later), which must not count as "missing".
    check_missing_values(df)
    verify_datetime(df)
    df = check_numeric_id(df)
    print(f"Validation passed for {len(df):,} rows")
    return df


## 6. Transform

In [ ]:
%%writefile transform.py
"""T — Transform: clean the validated data and add time columns for trend analysis."""
import pandas as pd

from prefect_compat import task


def remove_invalid_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows whose id was non-numeric (marked NaN during validation)."""
    df = df.dropna(subset=["id"]).copy()
    df["id"] = df["id"].astype("int64")
    return df


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """One row per id; if an incident appears twice keep the most recently updated version."""
    return (df.sort_values("last_updated", ascending=False, na_position="last")
              .drop_duplicates(subset=["id"], keep="first"))


def clean_text(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["type", "subtype", "location", "description"]:
        df[col] = df[col].astype("string").str.strip()
    df["type"] = df["type"].str.upper()
    return df


def split_datetime(df: pd.DataFrame) -> pd.DataFrame:
    df["date_time"] = pd.to_datetime(df["date_time"])
    df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")
    dt = df["date_time"].dt
    df["year"], df["month"], df["day"] = dt.year, dt.month, dt.day
    df["hour"], df["minute"], df["second"] = dt.hour, dt.minute, dt.second
    df["day_of_week"] = dt.day_name()
    return df


@task
def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = remove_invalid_rows(df)
    df = remove_duplicates(df)
    df = clean_text(df)
    df = split_datetime(df)
    df = df.sort_values("date_time").reset_index(drop=True)
    print(f"Transformed: {before:,} -> {len(df):,} rows")
    return df


## 7. Load into PostgreSQL (upsert)

In [ ]:
%%writefile load.py
"""L — Load: upsert the transformed data into PostgreSQL.

Unlike `df.to_sql(if_exists='replace')`, this keeps the table (and its PRIMARY KEY)
and inserts new incidents / updates changed ones, so daily runs are safe to repeat.
"""
import pandas as pd

from config import TABLE, get_connection
from prefect_compat import task

COLUMNS = ["id", "date_time", "type", "subtype", "location", "description", "last_updated",
           "year", "month", "day", "hour", "minute", "second", "day_of_week"]

CREATE_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    id           BIGINT PRIMARY KEY,
    date_time    TIMESTAMP NOT NULL,
    type         TEXT NOT NULL,
    subtype      TEXT,
    location     TEXT NOT NULL,
    description  TEXT,
    last_updated TIMESTAMP,
    year         INTEGER,
    month        INTEGER,
    day          INTEGER,
    hour         INTEGER,
    minute       INTEGER,
    second       INTEGER,
    day_of_week  TEXT
);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_date_time ON {TABLE} (date_time);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_type      ON {TABLE} (type);
"""

_updates = ", ".join(f"{c} = EXCLUDED.{c}" for c in COLUMNS if c != "id")
UPSERT_SQL = f"""
INSERT INTO {TABLE} ({", ".join(COLUMNS)}) VALUES %s
ON CONFLICT (id) DO UPDATE SET {_updates};
"""


def create_postgres_table(conn) -> None:
    with conn.cursor() as cur:
        cur.execute(CREATE_TABLE_SQL)
    conn.commit()


def _to_records(df: pd.DataFrame):
    """DataFrame -> list of tuples with NaN/NaT converted to None and numpy types to Python."""
    out = df[COLUMNS].astype(object).where(df[COLUMNS].notna(), None)
    records = []
    for row in out.itertuples(index=False):
        records.append(tuple(v.to_pydatetime() if isinstance(v, pd.Timestamp)
                             else (v.item() if hasattr(v, "item") else v) for v in row))
    return records


@task
def load_into_postgres(df: pd.DataFrame) -> int:
    from psycopg2.extras import execute_values

    conn = get_connection()
    try:
        create_postgres_table(conn)
        with conn.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {TABLE}")
            before = cur.fetchone()[0]
            execute_values(cur, UPSERT_SQL, _to_records(df), page_size=5000)
            cur.execute(f"SELECT COUNT(*) FROM {TABLE}")
            after = cur.fetchone()[0]
        conn.commit()
    finally:
        conn.close()
    print(f"Loaded {len(df):,} rows - {after - before:,} new incidents, {after:,} total in {TABLE}")
    return after


## 8. Monitor crime trends

In [ ]:
%%writefile monitor.py
"""Monitor: query PostgreSQL for crime trends, flag unusual spikes, save charts + a report.

Outputs (in REPORT_DIR, default ./reports):
  monthly_trend.png, by_hour.png, top_types.png, top_locations.png, trend_report.md
"""
import os
from datetime import datetime

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import TABLE, REPORT_DIR, get_connection
from prefect_compat import task

# ---------- SQL used for monitoring (also handy to paste into Metabase / pgAdmin) ----------
QUERIES = {
    "monthly": f"""
        SELECT date_trunc('month', date_time)::date AS month, COUNT(*) AS incidents
        FROM {TABLE} GROUP BY 1 ORDER BY 1""",
    "monthly_by_type": f"""
        SELECT date_trunc('month', date_time)::date AS month, type, COUNT(*) AS incidents
        FROM {TABLE} GROUP BY 1, 2 ORDER BY 1, 2""",
    "by_hour": f"""
        SELECT hour, COUNT(*) AS incidents FROM {TABLE} GROUP BY 1 ORDER BY 1""",
    "top_types": f"""
        SELECT type, COUNT(*) AS incidents FROM {TABLE}
        GROUP BY 1 ORDER BY 2 DESC LIMIT 10""",
    "top_locations_30d": f"""
        SELECT location, COUNT(*) AS incidents FROM {TABLE}
        WHERE date_time >= (SELECT MAX(date_time) FROM {TABLE}) - INTERVAL '30 days'
        GROUP BY 1 ORDER BY 2 DESC LIMIT 10""",
}

Z_THRESHOLD = 2.0     # flag a month that is > 2 standard deviations above its baseline
BASELINE_MONTHS = 12  # compare against the previous 12 months
MIN_BASELINE = 6      # need at least this many months of history to judge a spike


def run_query(conn, sql: str) -> pd.DataFrame:
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


def complete_months(monthly: pd.DataFrame, last_ts) -> pd.DataFrame:
    """Drop the current month if it's still in progress, so it isn't compared unfairly."""
    monthly = monthly.copy()
    monthly["month"] = pd.to_datetime(monthly["month"])
    last_ts = pd.Timestamp(last_ts)
    month_end = (last_ts + pd.offsets.MonthEnd(0)).normalize()
    if last_ts.normalize() < month_end:
        monthly = monthly[monthly["month"] < last_ts.to_period("M").to_timestamp()]
    return monthly


def detect_spikes(monthly_by_type: pd.DataFrame) -> pd.DataFrame:
    """For each incident type, z-score of the latest complete month vs the prior 12 months."""
    pivot = (monthly_by_type.pivot_table(index="month", columns="type",
                                         values="incidents", aggfunc="sum")
                            .sort_index().fillna(0))
    if len(pivot) < MIN_BASELINE + 1:
        return pd.DataFrame(columns=["type", "latest", "baseline_mean", "z_score"])
    latest = pivot.iloc[-1]
    base = pivot.iloc[-(BASELINE_MONTHS + 1):-1]
    mean, std = base.mean(), base.std(ddof=0).replace(0, float("nan"))
    z = (latest - mean) / std
    res = pd.DataFrame({"type": pivot.columns, "latest": latest.values,
                        "baseline_mean": mean.round(1).values, "z_score": z.round(2).values})
    return (res[(res["z_score"] > Z_THRESHOLD) & (res["latest"] >= 5)]
              .sort_values("z_score", ascending=False))


def _bar(df, x, y, title, path, horizontal=False):
    fig, ax = plt.subplots(figsize=(9, 5))
    if horizontal:
        d = df.iloc[::-1]
        ax.barh(d[x].astype(str), d[y], color="#3b6ea5")
    else:
        ax.bar(df[x].astype(str), df[y], color="#3b6ea5")
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


@task
def monitor_trends() -> dict:
    os.makedirs(REPORT_DIR, exist_ok=True)
    conn = get_connection()
    try:
        data = {name: run_query(conn, sql) for name, sql in QUERIES.items()}
        last_ts = run_query(conn, f"SELECT MAX(date_time) AS m FROM {TABLE}")["m"].iloc[0]
    finally:
        conn.close()

    monthly = complete_months(data["monthly"], last_ts)
    mbt = data["monthly_by_type"].copy()
    mbt["month"] = pd.to_datetime(mbt["month"])
    mbt = mbt[mbt["month"].isin(monthly["month"])]

    # month-over-month change + 3-month rolling average
    monthly["incidents"] = monthly["incidents"].astype(int)
    monthly["mom_change_pct"] = (monthly["incidents"].pct_change() * 100).round(1)
    monthly["rolling_3m"] = monthly["incidents"].rolling(3).mean().round(1)
    spikes = detect_spikes(mbt)

    # ---- charts ----
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(monthly["month"], monthly["incidents"], marker="o", ms=3, label="Monthly incidents")
    ax.plot(monthly["month"], monthly["rolling_3m"], lw=2, label="3-month average")
    ax.set_title("Reported incidents per month (Cambridge PD)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(REPORT_DIR, "monthly_trend.png"), dpi=120)
    plt.close(fig)
    _bar(data["by_hour"], "hour", "incidents", "Incidents by hour of day",
         os.path.join(REPORT_DIR, "by_hour.png"))
    _bar(data["top_types"], "type", "incidents", "Top 10 incident types (all time)",
         os.path.join(REPORT_DIR, "top_types.png"), horizontal=True)
    _bar(data["top_locations_30d"], "location", "incidents", "Top 10 locations (last 30 days)",
         os.path.join(REPORT_DIR, "top_locations.png"), horizontal=True)

    # ---- markdown report ----
    lines = [f"# Crime trend report — generated {datetime.now():%Y-%m-%d %H:%M}", "",
             f"Data through: {pd.Timestamp(last_ts):%Y-%m-%d %H:%M}", ""]
    if len(monthly):
        last = monthly.iloc[-1]
        lines += ["## Latest complete month",
                  f"- {last['month']:%B %Y}: **{last['incidents']:,}** incidents "
                  f"({last['mom_change_pct']:+.1f}% vs previous month)"
                  if pd.notna(last["mom_change_pct"]) else
                  f"- {last['month']:%B %Y}: **{last['incidents']:,}** incidents", ""]
    lines += ["## Spike alerts (z-score > %.1f vs previous %d months)" % (Z_THRESHOLD, BASELINE_MONTHS)]
    if spikes.empty:
        lines.append("- No unusual spikes detected.")
    else:
        for r in spikes.itertuples():
            lines.append(f"- **{r.type}**: {int(r.latest)} incidents "
                         f"(usual ≈ {r.baseline_mean}, z = {r.z_score})")
    peak_hour = data["by_hour"].sort_values("incidents").iloc[-1]["hour"] if len(data["by_hour"]) else None
    lines += ["", "## Other observations",
              f"- Busiest hour of the day: {int(peak_hour):02d}:00" if peak_hour is not None else "",
              "- Most common types: " + ", ".join(data["top_types"]["type"].head(3).astype(str)),
              "", "## Last 12 months", "",
              monthly.tail(12).assign(month=lambda d: d["month"].dt.strftime("%Y-%m"))
                     .to_markdown(index=False) if len(monthly) else "(no data)"]
    report_path = os.path.join(REPORT_DIR, "trend_report.md")
    with open(report_path, "w") as f:
        f.write("\n".join(lines) + "\n")

    for alert in spikes.itertuples():
        print(f"ALERT: spike in {alert.type} ({int(alert.latest)} vs ~{alert.baseline_mean})")
    print(f"Report written to {report_path}")
    return {"monthly": monthly, "spikes": spikes, **data}


## 9. The pipeline (Prefect flow)

In [ ]:
%%writefile etl_pipeline.py
"""Full pipeline: Extract -> Validate -> Transform -> Load -> Monitor.

    python etl_pipeline.py            # run once
    python etl_pipeline.py --serve    # register a Prefect deployment that runs daily at midnight
"""
import sys

from extract import extract_data
from validate import validate_data
from transform import transform_data
from load import load_into_postgres
from monitor import monitor_trends
from prefect_compat import flow, HAS_PREFECT


@flow(name="cpd_incident_etl", log_prints=True)  # shows as 'cpd_incident_etl' in the Prefect UI
def etl():
    print("Extracting data...")
    raw = extract_data()

    print("Performing data quality checks...")
    validated = validate_data(raw)

    print("Performing data transformations...")
    transformed = transform_data(validated)

    print("Importing data into Postgres...")
    load_into_postgres(transformed)

    print("Analysing crime trends...")
    monitor_trends()

    print("ETL complete!")


if __name__ == "__main__":
    if "--serve" in sys.argv:
        if not HAS_PREFECT:
            sys.exit("Install Prefect (pip install prefect) to schedule the pipeline.")
        # CPD data updates daily, so run every day at midnight
        etl.serve(name="cpd-pipeline-deployment", cron="0 0 * * *")
    else:
        etl()


## 10. Run step by step

In [ ]:
import importlib, config
importlib.reload(config)
from extract import extract_data
from validate import validate_data
from transform import transform_data
from load import load_into_postgres
from monitor import monitor_trends

raw = extract_data()
raw.head()

In [ ]:
validated = validate_data(raw)
transformed = transform_data(validated)
transformed.head()

In [ ]:
load_into_postgres(transformed)
!sudo -u postgres psql -d cpd_db -c "SELECT type, COUNT(*) FROM cpd_incidents GROUP BY 1 ORDER BY 2 DESC LIMIT 10;"

In [ ]:
results = monitor_trends()
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/trend_report.md').read()))
for f in ['monthly_trend', 'by_hour', 'top_types', 'top_locations']:
    display(Image(f'reports/{f}.png'))

## 11. Run the whole pipeline in one go
Re-running is safe: rows are upserted on `id`, so no duplicates are created.

In [ ]:
from etl_pipeline import etl
etl()

## 12. Scheduling (outside Colab)
On your own machine, `python etl_pipeline.py --serve` registers a Prefect deployment that runs daily at midnight. With `docker compose up -d` you also get pgAdmin (localhost:8081) and a Metabase dashboard (localhost:3000) — see README.md.